# AI-sfa 나카무라 Wan LoRA 평가 3판 — 여러 씨앗 + CPU 노이즈 기준선

2판에서 같은 씨앗이라도 PC와 Kaggle 결과가 구도부터 달랐다(GPU 난수 차이가 유력) → 씨앗 하나로는 판정할 수 없다.
3판: 초기 노이즈를 CPU에서 만들어 장비와 상관없이 재현, 프롬프트 7개(기존 4 + 학습 캡션 낱말판 번개·임팩트 고리 + 로봇) × 씨앗 3개 × (학습 전·세기 0.6·0.8) = 영상 63개.
재학습한 LoRA와 같은 씨앗으로 비교할 기준선이 된다.

**설정:**
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On**
3. (선택) **Add-ons → Secrets**에 `RCLONE_CONFIG_GDRIVE_TOKEN`(Claude 환경 변수와 같은 이름·값, 예전 `RCLONE_GDRIVE_AUTH_B64`도 됨)을 붙이면 결과가 Drive `AI-sfa/02_분석자료/WanLoRA_평가/`로 올라간다. 없으면 Output 탭에서 받는다.

**실행:** **Save Version → Save & Run All (Commit)**. 예상 60–90분(추정). 결과 `wan_out_eval3.zip`: `compare3/p{이름}_sheet.jpg`(줄 = 조건×씨앗), `p{이름}_grid.mp4`(3×3 나란히 재생), `samples3/`, `logs3/`.

In [ ]:
# 1. (선택) Drive 연결 — 비밀값이 없으면 건너뛴다. 인증값은 출력하지 않는다
import os, json, base64, pathlib, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    sc = UserSecretsClient()
    try:  # 사용자 Claude 환경 변수와 같은 이름: rclone 토큰 JSON 그대로(10-28)
        d = {'token': sc.get_secret('RCLONE_CONFIG_GDRIVE_TOKEN').strip()}
        json.loads(d['token'])
    except Exception:  # 예전 방식: client_id·secret·token을 담은 base64 JSON
        raw = sc.get_secret('RCLONE_GDRIVE_AUTH_B64').strip()
        d = json.loads(base64.b64decode(raw + '=' * (-len(raw) % 4)))
    p = pathlib.Path.home() / '.config/rclone'; p.mkdir(parents=True, exist_ok=True)
    lines = ['[gdrive]', 'type = drive', 'scope = drive.readonly,drive.file']
    if d.get('client_id'): lines.append('client_id = ' + d['client_id'])
    if d.get('client_secret'): lines.append('client_secret = ' + d['client_secret'])
    lines.append('token = ' + d['token'])
    (p / 'rclone.conf').write_text('\n'.join(lines) + '\n'); os.chmod(p / 'rclone.conf', 0o600)
    del d, lines
    subprocess.run('cd /tmp && curl -sSLo rc.zip https://downloads.rclone.org/rclone-current-linux-amd64.zip && unzip -qo rc.zip && cp rclone-*/rclone /usr/local/bin/', shell=True, check=True)
    print('Drive 연결됨')
except Exception as e:
    print('Drive 연결 없이 진행(결과는 Output 탭):', type(e).__name__)

In [ ]:
# 2. 작업자 파일 쓰기 → 실행(받기·설치 → 63개 GPU 나눠서 → 시트 → zip)
open('/tmp/kaggle_wan_eval3_worker.py', 'w', encoding='utf-8').write("#!/usr/bin/env python3\n\"\"\"Kaggle T4 x2: 나카무라 Wan LoRA 평가 3판 — 여러 씨앗 + CPU 노이즈 기준선 (2026-10-28 밤).\n\n2판에서 같은 씨앗이라도 PC(5060)와 T4의 결과가 구도부터 달랐다(GPU 난수 차이가 유력) → 씨앗 하나로는 판정 불가.\n3판: 초기 노이즈를 CPU에서(--cpu_noise, 장비 무관 재현), 프롬프트 7개 × 씨앗 3개 × (학습 전 · 세기 0.6 · 0.8) = 63개.\n프롬프트: 기존 p0–p3(pc_wan_lora_run.PROMPTS) + 학습 캡션 낱말로 고친 번개(1v)·임팩트 고리(3v) + 로봇(4r).\n  'fork'·'shockwave'는 학습 캡션에 한 번도 없다(277컷 집계) → 1v·3v는 학습 캡션에 실제로 나오는 낱말(branches, monochrome impact inserts, rings expand)로.\n  로봇은 지금 LoRA 캡션에 없는 낱말 — 인물·로봇 캡션을 넣은 재학습 뒤 같은 씨앗으로 다시 비교할 기준선.\nDiT fp16(T4에서 bf16과 결과 거의 같고 5.6배 빠름, 2판), T5 PC와 같게(fp8 저장 + bf16).\n도우미는 kaggle_wan_eval_worker.py(같은 브랜치)를 받아 쓴다.\n사용(노트북): python kaggle_wan_eval3_worker.py [--out /kaggle/working]\n\"\"\"\nimport argparse\nimport glob\nimport os\nimport subprocess\nimport sys\nimport threading\nimport zipfile\n\nW = os.environ.get(\"WAN_DIR\", \"/tmp/wan\")\nRAW = \"https://raw.githubusercontent.com/twoapple08/AI-sfa/ccr-bccf15a6-d6ee46/tools\"\nSEEDS = (11, 22, 33)\nEXTRA = {\n    \"1v\": \"nkmr_fx, Cyan-white lightning with a pink rim appears at 0-1 and branches into several bolts at 2-7. The lightning flickers on ones at 8-15, then a thin glowing afterimage fades through 32. Fixed camera, one cut.\",\n    \"3v\": \"nkmr_fx, Monochrome impact inserts alternate on ones at 0-3. Orange rings expand from the center at 4-15 with white radial lines, hold at 16-18, then break into fading streaks through 32. Fixed camera, one cut.\",\n    \"4r\": \"nkmr_fx, A giant robot punches at 0-3. A white-yellow impact flash bursts at its fist at 4-6, orange sparks and gray debris scatter at 7-20, and dark smoke rises through 32. Mostly twos; fixed camera, one cut.\",\n}\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--out\", default=\"/kaggle/working\")\n    ap.add_argument(\"--compare-only\", help=\"시험용: 이 폴더의 영상으로 시트만\")\n    a = ap.parse_args()\n    os.makedirs(W, exist_ok=True)\n    if not a.compare_only:\n        subprocess.run([\"curl\", \"-sSL\", \"-o\", f\"{W}/kaggle_wan_eval_worker.py\", f\"{RAW}/kaggle_wan_eval_worker.py\"], check=True)\n    sys.path.insert(0, W if not a.compare_only else os.path.dirname(os.path.abspath(__file__)))\n    import kaggle_wan_eval_worker as ev\n    conds = [\"base\", \"m0.6\", \"m0.8\"]\n    names = [\"0\", \"1\", \"2\", \"3\", \"1v\", \"3v\", \"4r\"]\n    if a.compare_only:\n        sheets(ev, a.compare_only, os.path.join(a.out, \"compare3\"), names, conds)\n        return\n    cap, ng, mt = ev.setup()\n    sys.path.insert(0, W)\n    import pc_wan_lora_run as pc\n    prompts = {str(i): p for i, p in enumerate(pc.PROMPTS)}\n    prompts.update(EXTRA)\n    sd = os.path.join(W, \"samples_v3\")\n    logdir = os.path.join(a.out, \"logs3\")\n    os.makedirs(sd, exist_ok=True)\n    os.makedirs(logdir, exist_ok=True)\n    ev.set_t5_dtype(mt, \"bfloat16\")\n    dit = os.path.join(W, \"models\", os.path.basename(ev.MODELS[\"dit_fp16\"][0]))\n    lora = pc.merge_ready(os.path.join(W, \"lora\", \"nkmr_wan13.safetensors\"), dit)\n    items = [(f\"{n}_s{s}\", prompts[n], 1000 * (k + 1) + s) for k, n in enumerate(names) for s in SEEDS]  # 묶음 안에서 씨앗이 겹치지 않게\n    half = len(items) // 2\n    jobs = []\n    for c in conds:\n        for part, its in ((\"a\", items[:half]), (\"b\", items[half:])):  # 조건마다 반씩 나눠 GPU 2장에 고르게\n            jobs.append(dict(tag=c, part=part, mult=float(c[1:]) if c != \"base\" else None, neg=None, items=its, dit=dit,\n                             lora=None if c == \"base\" else lora, mt=mt, sd=sd, logdir=logdir, cpu_noise=True))\n    ev.say(f\"평가 3판: 프롬프트 {len(names)} × 씨앗 {len(SEEDS)} × {conds} = {len(items) * len(conds)}개, GPU {ng}장\")\n    errs = ev.run_parallel(jobs, ng, threading.Lock())\n    sheets(ev, sd, os.path.join(a.out, \"compare3\"), names, conds)\n    zp = os.path.join(a.out, \"wan_out_eval3.zip\")\n    with zipfile.ZipFile(zp, \"w\", zipfile.ZIP_DEFLATED) as z:\n        for f in sorted(glob.glob(os.path.join(sd, \"*.mp4\"))):\n            z.write(f, \"samples3/\" + os.path.basename(f))\n        for d in (\"compare3\", \"logs3\"):\n            for f in sorted(glob.glob(os.path.join(a.out, d, \"*\"))):\n                z.write(f, os.path.relpath(f, a.out))\n        z.writestr(\"prompts3.txt\", \"\\n\".join(f\"{n}: {prompts[n]}\" for n in names) + \"\\n\")\n    n = len(glob.glob(os.path.join(sd, \"*.mp4\")))\n    ev.say(f\"완료: 영상 {n}/{len(items) * len(conds)}개 → {zp}\", \"| 실패:\" if errs else \"\", \"; \".join(errs))\n    if ev.shutil.which(\"rclone\") and os.path.exists(os.path.expanduser(\"~/.config/rclone/rclone.conf\")):\n        r = subprocess.run([\"rclone\", \"copy\", zp, \"gdrive:AI-sfa/02_분석자료/WanLoRA_평가/\"], capture_output=True, text=True)\n        ev.say(\"Drive 업로드:\", \"성공\" if r.returncode == 0 else \"실패 — Output 탭에서 받아 주세요\")\n\n\ndef sheets(ev, sd, outdir, names, conds):\n    \"\"\"프롬프트마다 (1) 줄 = 조건×씨앗, 칸 = 장 0·8·16·24·32 시트 JPG (2) 3×3 나란히 재생 mp4(열 = 조건, 줄 = 씨앗).\"\"\"\n    import cv2\n    import numpy as np\n    os.makedirs(outdir, exist_ok=True)\n    tw, th = 256, 144\n\n    def lab(img, s):\n        img = cv2.resize(img, (tw, th))\n        cv2.rectangle(img, (0, 0), (8 + 8 * len(s), 18), (0, 0, 0), -1)\n        cv2.putText(img, s, (3, 13), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)\n        return img\n\n    for n in names:\n        vids = {}\n        for c in conds:\n            for s in SEEDS:\n                f = os.path.join(sd, f\"p{n}_s{s}_{c}.mp4\")\n                if os.path.exists(f):\n                    vids[(c, s)] = ev.frames(f)\n        if not vids:\n            continue\n        rows = []\n        for c in conds:\n            for s in SEEDS:\n                v = vids.get((c, s))\n                if v:\n                    rows.append(np.hstack([lab(v[min(k, len(v) - 1)].copy(), f\"{c} s{s} f{k}\" if k == 0 else f\"f{k}\") for k in (0, 8, 16, 24, 32)]))\n        cv2.imwrite(os.path.join(outdir, f\"p{n}_sheet.jpg\"), np.vstack(rows), [cv2.IMWRITE_JPEG_QUALITY, 85])\n        nf = max(len(v) for v in vids.values())\n        vw = cv2.VideoWriter(os.path.join(outdir, f\"p{n}_grid.mp4\"), cv2.VideoWriter_fourcc(*\"mp4v\"), 24, (tw * len(conds), th * len(SEEDS)))\n        for k in range(nf):\n            grid = [[lab(vids[(c, s)][min(k, len(vids[(c, s)]) - 1)].copy(), f\"{c} s{s}\") if (c, s) in vids else np.zeros((th, tw, 3), np.uint8)\n                     for c in conds] for s in SEEDS]\n            vw.write(np.vstack([np.hstack(r) for r in grid]))\n        vw.release()\n\n\nif __name__ == \"__main__\":\n    main()\n")
!python -u /tmp/kaggle_wan_eval3_worker.py --out /kaggle/working